In [1]:
from config import MODEL_CONFIGS
from pathlib import Path
import pandas as pd
from datasets import Dataset
import json
from transformers import AutoTokenizer
from patching import extract_code_blocks, get_similarity, fix_code

WORK_DIR = Path.cwd()
DS_TEST_PATH = WORK_DIR / "dataset" / "split" / "test_dataset.jsonl"
ds_test = Dataset.from_pandas(pd.read_json(DS_TEST_PATH, lines=True))
id_to_idx = {item['id']: idx for idx, item in enumerate(ds_test)}

def compute_similarity(example: dict, result: dict, test_type: str):

    bad_code = example["bad_code"]
    good_code = example["good_code"]

    if test_type in ["fine_tuned, fine_tuned_cot"]: 
        # model is fine tuned and will output <FIX>
        
        fix = result["answer"]
        fixed_code = fix_code(bad_code, fix)

    else: 
        # model is not fine tuned and will give full repaired code
        blocks = extract_code_blocks(result["answer"])
        if not blocks:  # Model thinks code is correct
            fixed_code = bad_code
        else:
            fixed_code = blocks[-1] # assume last block is fixed code

    return get_similarity(good_code, fixed_code)


def present_results(model: str, test_type: str):

    if test_type not in ["baseline", "rag_only", "fine_tuned", "fine_tuned_cot"]:
        raise ValueError("Unknown test type: {test_type}")
    
    if model not in MODEL_CONFIGS:
        raise ValueError(
            f"Unknown model '{model}'. "
            f"Available models: {list(MODEL_CONFIGS.keys())}"
        )

    cfg = MODEL_CONFIGS[model]
    model_name = cfg["model_name"]
    model_short = model_name.split("/")[1]
        
    RESULTS_DIR = WORK_DIR / "results" / "testing" / model_short / test_type / "test_results.json"
    

    # Load results
    with open(RESULTS_DIR) as f:
        results = json.load(f)

    doms, syns, nones, toks = [], [], [], []

    editing = False
    tokenizer = None

    for result in results:

        id = result["id"] 
        answer = result["answer"]
        idx = id_to_idx[id]
        category = ds_test[idx]["mutation_category"]
        mut_type = ds_test[idx]["mutation_type"]

        passed = result.get("passed", None)
        token_len = result.get("token_len", None)
        
        if not passed: # value not computed already:

            if not editing:
                editing = True

            sim = compute_similarity(ds_test[idx], result, test_type)
            passed = 1 if sim == 1.0 else 0

            result["mutation_category"] = category
            result["mutation_type"] = mut_type
            result["similarity"] = sim
            result["passed"] = passed

        if not token_len:

            if not editing:
                editing = True

            if not tokenizer:
                tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
            token_len = len(tokenizer.encode(answer, add_special_tokens=True))
            result["token_len"] = token_len
        
        toks.append(token_len)

        if category == "domain":
            doms.append(passed)
        elif category == "syntax":
            syns.append(passed)
        elif category == "none":
            nones.append(passed)

    dom_score = sum(doms) / len(doms) if len(doms) != 0 else None
    syn_score = sum(syns) / len(syns) if len(syns) != 0 else None
    none_score = sum(nones) / len(nones) if len(nones) != 0 else None

    if dom_score and syn_score and none_score:
        overall_score = (dom_score+syn_score+none_score)/3
    else:
        overall_score = None

    tok_len_avg = int(sum(toks) / len(toks))

    print(f"Performance for {model_short}, {test_type}:")
    print(f"Syntax: {syn_score} (n={len(syns)})")
    print(f"Domain: {dom_score} (n={len(doms)})")
    print(f"Correct: {none_score} (n={len(nones)})")
    print(f"Overall: {overall_score}")
    print(f"Avg Generated Tokens: {tok_len_avg}\n")

    if editing:
        with open(RESULTS_DIR, 'w') as f:
            json.dump(results, f, indent=2)

In [2]:
present_results(model="qwen_coder_1p5b", test_type="baseline")

Performance for Qwen2.5-Coder-1.5B-Instruct, baseline:
Syntax: 0.2009987515605493 (n=801)
Domain: 0.006802721088435374 (n=147)
Correct: 0.18287037037037038 (n=432)
Overall: 0.13022394767311837
Avg Generated Tokens: 365



In [3]:
present_results(model="qwen_coder_1p5b", test_type="rag_only")

Performance for Qwen2.5-Coder-1.5B-Instruct, rag_only:
Syntax: None (n=0)
Domain: 0.0 (n=147)
Correct: 0.24305555555555555 (n=432)
Overall: None
Avg Generated Tokens: 349



In [4]:
present_results(model="qwen_coder_1p5b", test_type="fine_tuned_cot")

Performance for Qwen2.5-Coder-1.5B-Instruct, fine_tuned_cot:
Syntax: 0.003745318352059925 (n=801)
Domain: 0.006802721088435374 (n=147)
Correct: 1.0 (n=432)
Overall: 0.3368493464801651
Avg Generated Tokens: 62



In [ ]:
Overall performance for baseline is:
Domain: 0.0061 (n=163)
Syntax: 0.1759 (n=790)
None: 0.2706 (n=462)
Overall: 0.1509

Overall performance with RAG is:
Domain: 0.0798 (n=163)
Syntax: 0.1759 (n=790)
None: 0.3333 (n=462)
Overall: 0.1963

In [ ]:
losses, epochs = [], []
for result in results:
    loss = result.get("eval_loss", None)
    if not loss:
        continue
    epoch = result.get("epoch", None)
    
    losses.append(loss)
    epochs.append(epoch)

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.plot(epochs, losses, marker='o', linewidth=2, markersize=4, color='#2E86AB')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Evaluation Loss over Training Epochs', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()